# IMPORTS

In [811]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import re

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer

warnings.filterwarnings("ignore")

# setting the seed
RANDOM_STATE = 18
TEST_SIZE = 0.2
DATA_DIR = Path("../data")
TARGET_COL = "House"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# LOAD DATA

In [812]:
characters_path = f"{DATA_DIR}/Characters.csv"
df = pd.read_csv(characters_path, delimiter=';')

print("Shape:", df.shape)
df.head()

Shape: (140, 15)


,Id,Name,Gender,Job,House,Wand,Patronus,Species,Blood status,Hair colour,Eye colour,Loyalty,Skills,Birth,Death
0,1,Harry James Potter,Male,Student,Gryffindor,"11"" Holly phoenix feather",Stag,Human,Half-blood,Black,Bright green,Albus Dumbledore | Dumbledore's Army | Order o...,Parseltongue| Defence Against the Dark Arts | ...,31 July 1980,NaN
1,2,Ronald Bilius Weasley,Male,Student,Gryffindor,"12"" Ash unicorn tail hair",Jack Russell terrier,Human,Pure-blood,Red,Blue,Dumbledore's Army | Order of the Phoenix | Hog...,Wizard chess | Quidditch goalkeeping,1 March 1980,NaN
2,3,Hermione Jean Granger,Female,Student,Gryffindor,"10¾"" vine wood dragon heartstring",Otter,Human,Muggle-born,Brown,Brown,Dumbledore's Army | Order of the Phoenix | Hog...,Almost everything,"19 September, 1979",NaN
3,4,Albus Percival Wulfric Brian Dumbledore,Male,Headmaster,Gryffindor,"15"" Elder Thestral tail hair core",Phoenix,Human,Half-blood,Silver| formerly auburn,Blue,Dumbledore's Army | Order of the Phoenix | Hog...,Considered by many to be one of the most power...,Late August 1881,"30 June, 1997"
4,5,Rubeus Hagrid,Male,Keeper of Keys and Grounds | Professor of Care...,Gryffindor,"16"" Oak unknown core",NaN,Half-Human/Half-Giant,Part-Human (Half-giant),Black,Black,Albus Dumbledore | Order of the Phoenix | Hogw...,Resistant to stunning spells| above average st...,6 December 1928,NaN


In [813]:
df = df.dropna(subset=[TARGET_COL])
print(f"Shape después de eliminar NaN en {TARGET_COL}:", df.shape)

Shape después de eliminar NaN en House: (101, 15)


In [814]:
counts = df[TARGET_COL].value_counts()
df = df[df[TARGET_COL].isin(counts[counts > 1].index)]
print(f"Shape después de eliminar clases poco frecuentes en {TARGET_COL}:", df.shape)

Shape después de eliminar clases poco frecuentes en House: (100, 15)


In [815]:
spells_path = f"{DATA_DIR}/Spells.csv"
df_spells = pd.read_csv(spells_path, delimiter=';')

print("Shape:", df_spells.shape)
df_spells.head().to_csv()

Shape: (301, 5)


',Name,Incantation,Type,Effect,Light\r\n0,Summoning Charm,Accio,Charm,Summons an object,\r\n1,Age Line,Unknown,Charm,Prevents people above or below a certain age from access to a target,Blue\r\n2,Water-Making Spell,Aguamenti,"Charm, Conjuration",Conjures\xa0water,Icy blue\r\n3,Launch an object up into the air,Alarte Ascendare,Charm,Rockets target upward,Red\r\n4,Albus Dumbledore\'s Forceful Spell,Unknown,Spell,Great Force,\r\n'

In [816]:
hp1_path = f"{DATA_DIR}/Harry Potter 1.csv"
df_hp1 = pd.read_csv(hp1_path, delimiter=';')

print("Shape:", df_hp1.shape)
df_hp1.head().to_csv()

Shape: (1587, 2)


',Character,Sentence\r\n0,Dumbledore,"I should\'ve known that you would be here, Professor McGonagall."\r\n1,McGonagall,"Good evening, Professor Dumbledore."\r\n2,McGonagall,"Are the rumors true, Albus?"\r\n3,Dumbledore,"I\'m afraid so, professor."\r\n4,Dumbledore,The good and the bad.\r\n'

In [817]:
hp2_path = f"{DATA_DIR}/Harry Potter 2.csv"
df_hp2 = pd.read_csv(hp2_path, delimiter=';')

print("Shape:", df_hp2.shape)
df_hp2.head().to_csv()

Shape: (1700, 2)


',Character,Sentence\r\n0,HARRY ,"I can’t let you out, Hedwig. "\r\n1,HARRY ,I’m not allowed to use magic outside of school. \r\n2,HARRY ,"Besides, if Uncle Vernon…"\r\n3,VERNON,Harry Potter!\r\n4,HARRY,Now you’ve done it.\r\n'

In [818]:
hp3_path = f"{DATA_DIR}/Harry Potter 3.csv"
df_hp3 = pd.read_csv(hp3_path, delimiter=';')

print("Shape:", df_hp3.shape)
df_hp3.head().to_csv()

Shape: (1638, 2)


',CHARACTER,SENTENCE\r\n0,HARRY,Lumos Maxima...\r\n1,HARRY,Lumos Maxima...\r\n2,HARRY,Lumos Maxima...\r\n3,HARRY,Lumos... MAXIMA!\r\n4,AUNT PETUNIA,Harry! Harry!\r\n'

In [819]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (80, 14) Test shape: (20, 14)


# FEATURE ENGINEERING FUNCTIONS

In [820]:
def handle_unknown_spell_incantation(spells_df: pd.DataFrame) -> pd.DataFrame:
    spells_df = spells_df.copy()
    spells_df['Incantation'] = spells_df.apply(
        lambda row: row['Name']
        if pd.isna(row['Incantation']) or str(row['Incantation']).strip().lower() == 'unknown'
        else row['Incantation'],
        axis=1
    )
    return spells_df

In [821]:
def to_lower_every_string_feature(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        if df[col].dtype == 'str':
            df[col] = df[col].astype(str).str.lower().str.strip()
    return df

def split_names_in_main_df(df: pd.DataFrame) -> object:
    df = df.copy()
    if 'Name' in df.columns:
        name_split = df['Name'].str.split()
    return name_split

def split_types_of_spells(spells_df: pd.DataFrame) -> pd.Series:
    spells_df = spells_df.copy()
    if 'Type' not in spells_df.columns:
        return pd.Series(dtype=object)

    spell_types_split = (
        spells_df['Type']
        .astype(str)
        .str.split(',')
        .apply(lambda items: [item.strip() for item in items if item.strip()])
        .explode()
        .drop_duplicates()
        .reset_index(drop=True)
    )
    return spell_types_split

def add_one_if_characher_said_something(df: pd.DataFrame, quotes_df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df_split_names = split_names_in_main_df(df)
    df['number_of_times_character_spoke'] = 0
    for name in df_split_names:
        for part in name:
            for char in quotes_df['Character']:
                if part in char:
                    df.loc[df['Name'].str.contains(part, case=False, na=False), 'number_of_times_character_spoke'] += 1
                
    return df

def check_what_spells_characters_mentioned_and_add_one_to_spell_type_counter(df: pd.DataFrame, spells_df: pd.DataFrame, quotes_df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df_split_names = split_names_in_main_df(df)
    spells_df = spells_df.copy()
    spells_split_types = split_types_of_spells(spells_df)
    quotes_df = quotes_df.copy()
    
    for type_ in spells_split_types.unique():
        df[f'mentioned_{type_}'] = 0
    
    for row in quotes_df.itertuples():
        for spell in spells_df.itertuples():
            Type = spell.Type
            if spell.Incantation in row.Sentence:
                types_of_spell = [t.strip() for t in spell.Type.split(',') if t.strip()]
                for name in df_split_names:
                    for part in name:
                        if part in row.Character:
                            # Para cada tipo del spell, marcar 1 en el feature correspondiente
                            for type_ in types_of_spell:
                                df.loc[df['Name'].str.contains(part, case=False, na=False), f'mentioned_{type_}'] += 1
    return df

In [822]:
df_hp3 = df_hp3.rename(columns={"CHARACTER": "Character", "SENTENCE": "Sentence"})

In [823]:
quotes_df = pd.concat([df_hp1, df_hp2, df_hp3], ignore_index=True)
quotes_df = to_lower_every_string_feature(quotes_df)
quotes_df.loc[quotes_df['Character'] == 'lee jordan', 'Character'] = 'lee'
quotes_df.loc[quotes_df['Character'] == 'ron', 'Character'] = 'ronald'

In [824]:
df = to_lower_every_string_feature(df)
df_spells = handle_unknown_spell_incantation(df_spells)
df_spells = to_lower_every_string_feature(df_spells)

df_names_split = split_names_in_main_df(df)
df_spell_types_split = split_types_of_spells(df_spells)


df = add_one_if_characher_said_something(df, quotes_df)
df = check_what_spells_characters_mentioned_and_add_one_to_spell_type_counter(df, df_spells, quotes_df)

display(df.head())

,Id,Name,Gender,Job,House,Wand,Patronus,Species,Blood status,Hair colour,Eye colour,Loyalty,Skills,Birth,Death,number_of_times_character_spoke,mentioned_charm,mentioned_conjuration,mentioned_spell,mentioned_healing spell,mentioned_vanishment,mentioned_hex,mentioned_jinx,mentioned_curse,mentioned_transportation,mentioned_transfiguration,mentioned_transformation,mentioned_dark charm,mentioned_counter-charm,mentioned_transfiguration jinx,mentioned_transfiguration spell,mentioned_counter-spell,mentioned_dark arts,mentioned_counter-jinx,mentioned_patented charm,mentioned_counter-curse,mentioned_untransfiguration
0,1,harry james potter,male,student,gryffindor,"11"" holly phoenix feather",stag,human,half-blood,black,bright green,albus dumbledore | dumbledore's army | order o...,parseltongue| defence against the dark arts | ...,31 july 1980,NaN,1037,20,0,2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,2,ronald bilius weasley,male,student,gryffindor,"12"" ash unicorn tail hair",jack russell terrier,human,pure-blood,red,blue,dumbledore's army | order of the phoenix | hog...,wizard chess | quidditch goalkeeping,1 march 1980,NaN,1562,15,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
2,3,hermione jean granger,female,student,gryffindor,"10¾"" vine wood dragon heartstring",otter,human,muggle-born,brown,brown,dumbledore's army | order of the phoenix | hog...,almost everything,"19 september, 1979",NaN,486,9,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0
3,4,albus percival wulfric brian dumbledore,male,headmaster,gryffindor,"15"" elder thestral tail hair core",phoenix,human,half-blood,silver| formerly auburn,blue,dumbledore's army | order of the phoenix | hog...,considered by many to be one of the most power...,late august 1881,"30 june, 1997",239,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,5,rubeus hagrid,male,keeper of keys and grounds | professor of care...,gryffindor,"16"" oak unknown core",NaN,half-human/half-giant,part-human (half-giant),black,black,albus dumbledore | order of the phoenix | hogw...,resistant to stunning spells| above average st...,6 december 1928,NaN,394,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [825]:
# Aplicar las mismas transformaciones de feature engineering a X_train y X_test
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

# Aplicar to_lower_every_string_feature
X_train_fe = to_lower_every_string_feature(X_train_fe)
X_test_fe = to_lower_every_string_feature(X_test_fe)

# Aplicar add_one_if_characher_said_something
X_train_fe = add_one_if_characher_said_something(X_train_fe, quotes_df)
X_test_fe = add_one_if_characher_said_something(X_test_fe, quotes_df)

# Aplicar check_what_spells_characters_mentioned_and_add_one_to_spell_type_counter
X_train_fe = check_what_spells_characters_mentioned_and_add_one_to_spell_type_counter(X_train_fe, df_spells, quotes_df)
X_test_fe = check_what_spells_characters_mentioned_and_add_one_to_spell_type_counter(X_test_fe, df_spells, quotes_df)

# Mostrar resultados
print("X_train_fe:")
display(X_train_fe.head())
print("X_test_fe:")
display(X_test_fe.head())

X_train_fe:


,Id,Name,Gender,Job,Wand,Patronus,Species,Blood status,Hair colour,Eye colour,Loyalty,Skills,Birth,Death,number_of_times_character_spoke,mentioned_charm,mentioned_conjuration,mentioned_spell,mentioned_healing spell,mentioned_vanishment,mentioned_hex,mentioned_jinx,mentioned_curse,mentioned_transportation,mentioned_transfiguration,mentioned_transformation,mentioned_dark charm,mentioned_counter-charm,mentioned_transfiguration jinx,mentioned_transfiguration spell,mentioned_counter-spell,mentioned_dark arts,mentioned_counter-jinx,mentioned_patented charm,mentioned_counter-curse,mentioned_untransfiguration
20,21,oliver wood,male,student,unknown,unknown,human,pure-blood or half-blood,NaN,NaN,hogwarts school of witchcraft and wizardry,keeper| captain of the gryffindor quidditch team,october 1975 - 31 august 1976,NaN,38,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
126,127,edward remus lupin,male,student,unknown,unknown,human (metamorphmagus),half-blood,variable,variable,NaN,NaN,"april, 1998",NaN,414,12,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
82,83,newton scamander,male,employee in the beast division of the departme...,unknown,unknown,human,pure-blood or half-blood,red brown,blue,NaN,"magizoology, order of merlin, second class",24 february 1897,NaN,168,4,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
85,86,zacharias smith,male,student,unknown,unknown,human,pure-blood or half-blood,blonde,NaN,dumbledore's army,chaser,1 september 1979 - 2 may 1981,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
30,31,molly weasley,female,NaN,unknown,unknown,human,pure-blood,red,bright brown,order of the phoenix,household spells| healing magic,"30 october, 1949",NaN,798,7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


X_test_fe:


,Id,Name,Gender,Job,Wand,Patronus,Species,Blood status,Hair colour,Eye colour,Loyalty,Skills,Birth,Death,number_of_times_character_spoke,mentioned_charm,mentioned_conjuration,mentioned_spell,mentioned_healing spell,mentioned_vanishment,mentioned_hex,mentioned_jinx,mentioned_curse,mentioned_transportation,mentioned_transfiguration,mentioned_transformation,mentioned_dark charm,mentioned_counter-charm,mentioned_transfiguration jinx,mentioned_transfiguration spell,mentioned_counter-spell,mentioned_dark arts,mentioned_counter-jinx,mentioned_patented charm,mentioned_counter-curse,mentioned_untransfiguration
36,37,filius flitwick,male,professor of charms | head of ravenclaw,unknown,non-corporeal,human(goblin ancestry),part-goblin,white,NaN,NaN,charms,17 october 1958 or earlier,NaN,13,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
24,25,lavender brown,female,student,unknown,unknown,human,pure-blood,blond,blue,dumbledore's army |hogwarts school of witchcra...,divination,1 september 1979- 31 august 1980,"2 may, 1998",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15,16,peter pettigrew,male,the servant of lord voldemort,"9¼"" chestnut dragon heartstring",NaN,human,half-blood or pure-blood,colourless and balding,blue,lord voldemort| death eaters,animagus,1 september 1959- 31 august 1960,late march 1998,19,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
56,57,blaise zabini,male,student,unknown,unknown,human,pure-blood or half-blood,black,brown,NaN,chaser,"1 september, 1979 – 21 april, 1980",NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
67,68,salazar slytherin,male,founder of slytherin,"snakewood, basilisk horn",unknown,human,pure-blood,grey,dark,NaN,accomplished legilimens and one of the first r...,pre 976,11th century,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
